# 切割下料问题

**类别:** 装箱

来源: [https://www.hexaly.com/templates/cutting-stock-problem](https://www.hexaly.com/templates/cutting-stock-problem)


## 问题描述

在切割下料问题中,我们必须从具有固定尺寸的大卷材(或板材)上切割出宽度和高度各不相同的矩形物品。切割遵循一个结构化的两级切断模式:

- **条带 (Bands)**:具有相同宽度的物品沿其高度方向并排排列,形成水平条带。每个条带具有固定的宽度(即该物品的共同宽度),其总长度等于所有物品高度之和。
- **卷材 (Rolls)**:条带随后被垂直堆叠在卷材内。一个卷材内所有条带的总宽度不能超过卷材的宽度。同样地,每个条带的长度必须能放入卷材的长度内。

目标是最小化所使用的卷材数量。

	

### 学到的要点

- 使用 **两层集合决策变量** 来建模一个分层装箱结构(物品放入条带,条带放入卷材)
- 对集合使用 **distinct 算子** 配合一个 lambda,以强制一个条带内的所有物品共享相同的宽度
- **用三元运算符保护对集合的 max 操作**(count(s) > 0 ? max(s, ...) : 0)以避免空集上的 NaN 问题


## 数据

我们提供的切割下料实例来自 [2DPackLib](https://site.unibo.it/operations-research/en/research/2dpacklib) 中的 A 类实例。数据文件的格式如下:

- 第一行:物品种类数
- 第二行:卷材宽度、卷材长度
- 对每种物品类型:宽度、高度、需求


## 模型

切割下料问题的 Hexaly 模型使用两层 [集合决策变量](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html)。在第一层,对于每个条带,我们定义一个集合变量来表示分配到该条带的物品。在第二层,对于每个卷材,我们定义一个集合变量来表示分配到该卷材的条带。两层都使用 partition 约束以确保每个物品恰好属于一个条带,且每个条带恰好属于一个卷材。

在每个条带内,所有物品必须共享相同的宽度。这通过 distinct 算子配合一个 lambda 函数来强制:`distinct(bands[b], (i) => rawWidth[i])` 返回该条带中不同的宽度集合,我们将其数量约束为不超过 1。我们使用可变参 sum 算子将物品高度累加作为条带长度,并约束每个条带的长度不超过卷材长度。在卷材层,我们使用可变参 sum 算子对卷材的条带集合配合返回每个条带宽度的 lambda 函数来计算条带的总宽度,并约束其不超过卷材宽度。

模型对卷材数量计算一个简单的下界(物品总面积除以卷材面积),并使用 hxObjectiveThreshold 在达到该下界时提前停止。


## Python 实现


In [ ]:
# Copyright (c) Hexaly. Permission is hereby granted to use, copy,
# and modify this code for applications developed with Hexaly.
import hexaly.optimizer
import sys
import math


def read_integers(filename):
    with open(filename) as f:
        return [int(elem) for elem in f.read().split()]

def main(instance_file, str_time_limit, output_file):
    #
    # Read instance data
    #
    file_it = iter(read_integers(instance_file))

    nb_item_types = int(next(file_it))
    roll_width = int(next(file_it))
    roll_length = int(next(file_it))

    # Read item types and expand by demand
    raw_width = []
    raw_length = []
    for t in range(nb_item_types):
        width = int(next(file_it))
        length = int(next(file_it))
        demand = int(next(file_it))
        for c in range(demand):
            raw_width.append(width)
            raw_length.append(length)

    nb_items = len(raw_width)

    nb_max_bands = nb_items
    nb_max_rolls = nb_items

    # Lower bound on total number of rolls: total item area / rollArea
    total_area = sum(raw_width[i] * raw_length[i] for i in range(nb_items))
    min_number_of_rolls = int(math.ceil(total_area / (roll_width * roll_length)))
        
    with hexaly.optimizer.HexalyOptimizer() as optimizer:
        #
        # Declare the optimization model
        #
        model = optimizer.model

        # Set decisions: bands[b] represents the items assigned to band b
        bands = [model.set(nb_items) for _ in range(nb_max_bands)]
        # Set decisions: rolls[r] represents the bands assigned to roll r
        rolls = [model.set(nb_max_bands) for _ in range(nb_max_rolls)]

        # Each item must be in exactly one band, and each band in exactly one roll
        model.constraint(model.partition(bands))
        model.constraint(model.partition(rolls))

        # Create model arrays for item data
        widths = model.array(raw_width)
        lengths = model.array(raw_length)

        band_width = [None for _ in range(nb_max_bands)]
        band_length = [None for _ in range(nb_max_bands)]

        for b in range(nb_max_bands):
            # Length constraint for each band
            band_length[b] = model.sum(
                bands[b], model.lambda_function(lambda i: lengths[i])
            )
            model.constraint(band_length[b] <= roll_length)

            # All items in a band must have the same width
            distinct_widths = model.distinct(
                bands[b], model.lambda_function(lambda i: widths[i])
            )
            model.constraint(model.count(distinct_widths) <= 1)

            band_width[b] = model.iif(
                model.count(bands[b]) > 0,
                model.max(bands[b], model.lambda_function(lambda i: widths[i])),
                0,
            )

        # Create array of band width expressions for roll lambdas
        band_width_array = model.array(band_width)

        roll_used = [None for _ in range(nb_max_rolls)]
        total_width = [None for _ in range(nb_max_rolls)]

        for r in range(nb_max_rolls):
            # The roll's width is the sum of the widths of its bands
            total_width[r] = model.sum(
                rolls[r], model.lambda_function(lambda b: band_width_array[b])
            )
            model.constraint(total_width[r] <= roll_width)

            roll_used[r] = model.count(rolls[r]) > 0

        # Minimize the number of used rolls
        number_of_used_rolls = model.sum(roll_used)
        model.minimize(number_of_used_rolls)
        model.close()

        # Parameterize the optimizer
        optimizer.param.time_limit = int(str_time_limit)

        # Stop the search if the lower threshold is reached
        optimizer.param.set_objective_threshold(0, min_number_of_rolls)

        optimizer.solve()

        # Write the solution in a file
        if output_file is not None:
            with open(output_file, "w") as f:
                f.write("Number of rolls used: %d\n" % number_of_used_rolls.value)
                f.write("Roll dimensions: %d x %d\n\n" % (roll_length, roll_width))
                roll_index = 0
                for r in range(nb_max_rolls):
                    if rolls[r].value.count() == 0:
                        continue
                    f.write("Roll %d | width: %d/%d\n"
                        % (roll_index, total_width[r].value, roll_width))
                    for b in rolls[r].value:
                        if bands[b].value.count() == 0:
                            continue
                        f.write("  Band width=%d length=%d | Items: "
                            % (band_width[b].value, band_length[b].value))
                        for i in bands[b].value:
                            f.write("%d " % i)
                        f.write("\n")
                    roll_index += 1

if __name__ == '__main__':
    if len(sys.argv) < 2:
        print("Usage: python cutting_stock.py inputFile [output_file] [time_limit]")
        sys.exit(1)

    instance_file = sys.argv[1]
    output_file = sys.argv[2] if len(sys.argv) > 2 else None
    str_time_limit = sys.argv[3] if len(sys.argv) > 3 else "10"
    main(instance_file, str_time_limit, output_file)
